Cite this paper if this code helps you:
F. Satter and S. M. Masudul Ahsan, "Open Vocabulary Word Recognition From Transcribed Bangla Texts," 2023 26th International Conference on Computer and Information Technology (ICCIT), Cox's Bazar, Bangladesh, 2023, pp. 1-6, doi: 10.1109/ICCIT60459.2023.10441393.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/gdrive')
    BASE_DIR = Path('/content/gdrive/MyDrive')
else:
    BASE_DIR = Path.cwd()

# Change these paths to your original files. The models/configs are not in this repository.
MODELS_RESEARCH_DIR = BASE_DIR / 'models' / 'research'
CUSTOM_TF2_DIR = BASE_DIR / 'customTF2'
DATA_DIR = CUSTOM_TF2_DIR / 'data'
SSD_MODEL_DIR = DATA_DIR / 'inference_graph'
FASTER_RCNN_MODEL_DIR = DATA_DIR / 'inference_graph (1)'
PATH_TO_LABELS = str(DATA_DIR / 'label_map.pbtxt')
PATH_TO_TEST_IMAGES_DIR = DATA_DIR / 'Test'
OUTPUT_DIR = BASE_DIR / 'runs' / 'SSD_Test_Result'
INSTALL_OBJECT_DETECTION = False  # One-time setup only; see README.md.
RUN_WORD_EVALUATION = False
EXPERIMENT = 'full'  # full, no_pno, or no_nms_pno (single models only)
if EXPERIMENT not in {'full', 'no_pno', 'no_nms_pno'}:
    raise ValueError('Unsupported EXPERIMENT for this notebook.')
_saved_models = {}

In [ ]:
if INSTALL_OBJECT_DETECTION:
    import shutil
    if not (MODELS_RESEARCH_DIR / 'object_detection' / 'packages' / 'tf2' / 'setup.py').is_file():
        raise FileNotFoundError(f"Provide the original TensorFlow models checkout: {MODELS_RESEARCH_DIR}")
    if shutil.which('protoc') is None:
        raise RuntimeError('Install the Protocol Buffers compiler (protoc) before API setup.')
    subprocess.run(['protoc', *[str(p.relative_to(MODELS_RESEARCH_DIR)) for p in
                   sorted((MODELS_RESEARCH_DIR / 'object_detection' / 'protos').glob('*.proto'))],
                   '--python_out=.'], cwd=MODELS_RESEARCH_DIR, check=True)
    shutil.copyfile(MODELS_RESEARCH_DIR / 'object_detection/packages/tf2/setup.py',
                    MODELS_RESEARCH_DIR / 'setup.py')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '.'], cwd=MODELS_RESEARCH_DIR, check=True)
    print('API installed. Restart the kernel before proceeding; set INSTALL_OBJECT_DETECTION=False.')
else:
    try:
        import object_detection
    except ImportError as exc:
        raise ImportError('TensorFlow Object Detection API is missing. Follow README.md, install the original checkout, and restart.') from exc

In [ ]:
import glob
import xml.etree.ElementTree as ET
import pandas as pd
import tensorflow as tf

# Configure GPUs once, before loading any SavedModel.
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    if not tf.config.experimental.get_memory_growth(gpu):
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
if (MODELS_RESEARCH_DIR / 'object_detection/builders/model_builder_tf2_test.py').is_file():
    subprocess.run([sys.executable, str(MODELS_RESEARCH_DIR / 'object_detection/builders/model_builder_tf2_test.py')], check=True)

In [ ]:
import cv2
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
import itertools

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    def cv2_imshow(image):
        from IPython.display import display
        if image.ndim == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(image))

In [ ]:
def validate_detections(detections):
    num_detections = int(detections['num_detections'][0])
    if num_detections < 0:
        raise ValueError('num_detections must be nonnegative.')
    for key in ('detection_boxes', 'detection_classes', 'detection_scores'):
        if detections[key].shape[1] < num_detections:
            raise ValueError(f'SavedModel output is shorter than num_detections: {key}')
    scores = detections['detection_scores'][0, :num_detections].numpy()
    boxes = detections['detection_boxes'][0, :num_detections].numpy()
    classes = detections['detection_classes'][0, :num_detections].numpy()
    if not np.all(np.isfinite(scores)) or not np.all(np.isfinite(boxes)):
        raise ValueError('SavedModel returned non-finite scores/boxes.')
    if np.any(scores[:-1] < scores[1:]):
        raise ValueError('The original method requires detections sorted by descending confidence.')
    if np.any(boxes[:, 2:] <= boxes[:, :2]):
        raise ValueError('SavedModel returned a zero-area or inverted box.')
    if np.any((classes < 1) | (classes > 92) | (classes != np.floor(classes))):
        raise ValueError('SavedModel must use the original class IDs 1..92.')

In [ ]:
classes=['blank','অ','ই','ঈ','উ','ঊ','ঋ','এ','ঐ','ও','ঔ','ক','খ','গ','ঘ','ঙ','চ','ছ','জ','ঝ','ঞ','ট','ঠ','ড','ঢ','ণ','ত','থ','দ','ধ','ন','প','ফ','ব','ভ','ম','য','র','ল','শ','ষ','স','হ','ড়','ঢ়','য়','ৎ','ঃ','ং','ঁ','০','১','২','৩','৪','৫','৬','৭','৮','৯','া','ি','ী','ে','ু','faka','্র','্য','ক্ষ','ন্ত','ত্র','ঙ্গ','স্থ','স্ব','ক্ত','স্ত','ন্দ','চ্ছ','দ্ধ','ন্ত্র','ফাকা','ত্ত','ষ্ট','ন্ন','ল্প','ম্প','faka','ূ','ৃ','ৈ','faka','ৌ','।']
labels=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92]
print(len(classes))
print(len(labels))

Main steps - including (7)NMS and (8)PNO

In [ ]:
if EXPERIMENT == 'full':
    """
    Object Detection (On Image) From TF2 Saved Model
    =====================================
    """
    final_Lekha = []
    def detect_frame(frame,IMAGE_PATHS,isRealTime = False):
        import os
        os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'    # Suppress TensorFlow logging (1)
        import pathlib
        import tensorflow as tf
        import cv2
        import argparse
        import numpy as np


        # PROVIDE PATH TO IMAGE DIRECTORY
        print(IMAGE_PATHS)
        IMAGE_PATHS = f'{IMAGE_PATHS}'

        if frame is None:
            raise ValueError(f"Cannot read image: {IMAGE_PATHS}")
        image_np = np.array(frame)
        # PROVIDE PATH TO MODEL DIRECTORY
        PATH_TO_MODEL_DIR = str(SSD_MODEL_DIR)

        # PROVIDE PATH TO LABEL MAP

        # PROVIDE THE MINIMUM CONFIDENCE THRESHOLD
        MIN_CONF_THRESH = float(0.20)

        # LOAD THE MODEL

        import time
        from object_detection.utils import label_map_util
        from object_detection.utils import visualization_utils as viz_utils
        PATH_TO_SAVED_MODEL = str(Path(PATH_TO_MODEL_DIR) / 'saved_model')

        print('Loading model...', end='')
        start_time = time.time()

        # LOAD SAVED MODEL AND BUILD DETECTION FUNCTION
        if PATH_TO_SAVED_MODEL not in _saved_models:
            if not (Path(PATH_TO_SAVED_MODEL) / 'saved_model.pb').is_file():
                raise FileNotFoundError(f'Provide the trained SavedModel: {PATH_TO_SAVED_MODEL}')
            _saved_models[PATH_TO_SAVED_MODEL] = tf.saved_model.load(PATH_TO_SAVED_MODEL)
        detect_fn = _saved_models[PATH_TO_SAVED_MODEL]

        end_time = time.time()
        elapsed_time = end_time - start_time
        print('Done! Took {} seconds'.format(elapsed_time))

        # LOAD LABEL MAP DATA FOR PLOTTING

        category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS,
                                                                            use_display_name=True)

        import numpy as np
        from PIL import Image
        import matplotlib.pyplot as plt
        import warnings


        def load_image_into_numpy_array(path):
            """Load an image from file into a numpy array.
            Puts image into numpy array to feed into tensorflow graph.
            Note that by convention we put it into a numpy array with shape
            (height, width, channels), where channels=3 for RGB.
            Args:
              path: the file path to the image
            Returns:
              uint8 numpy array with shape (img_height, img_width, 3)
            """
            return np.array(Image.open(path))

        print('Running inference for {}... '.format(IMAGE_PATHS), end='')

        image = cv2.imread(IMAGE_PATHS)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {IMAGE_PATHS}")
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_expanded = np.expand_dims(image_rgb, axis=0)

        # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
        # Preserve original BGR input. Binary word images have identical channels.
        input_tensor = tf.convert_to_tensor(image)
        # The model expects a batch of images, so add an axis with `tf.newaxis`.
        input_tensor = input_tensor[tf.newaxis, ...]

        detections = detect_fn(input_tensor)
        validate_detections(detections)
        temp__num_detection = 0
        for i in range(min(100, int(detections['num_detections'][0]))):
          if detections['detection_scores'][0][i]>0.2 :
            temp__num_detection+=1
        print(temp__num_detection)

        detections['num_detections'] = temp__num_detection

        # All outputs are batches tensors.
        # Convert to numpy arrays, and take index [0] to remove the batch dimension.
        # We're only interested in the first num_detections.
        num_detections = int(detections.pop('num_detections'))

        detections = {key: value[0, :num_detections].numpy()
                      for key, value in detections.items()}
        detections['num_detections'] = num_detections
        print('hoye ja')
        print(detections['detection_classes'])

        # detection_classes should be ints.
        detections['detection_classes'] = detections['detection_classes'].astype(np.int64)

        image_with_detections = image.copy()

        # SET MIN_SCORE_THRESH BASED ON YOU MINIMUM THRESHOLD FOR DETECTIONS
        viz_utils.visualize_boxes_and_labels_on_image_array(
              image_with_detections,
              detections['detection_boxes'],
              detections['detection_classes'],
              detections['detection_scores'],
              category_index,
              use_normalized_coordinates=True,
              max_boxes_to_draw=100,
              min_score_thresh=0.2,
              agnostic_mode=False)

        print('Done')
        print(num_detections)

        print(detections['detection_classes'])
        print(detections['detection_scores'])
        cv2_imshow(image_with_detections)

        mark = [0]*num_detections
        myletters = []
        for i in range(0,num_detections):
            cur=detections['detection_classes'][i]
            current_label = labels[cur]
            if(current_label == 86):
              current_label = 64
            print('CURRENT_LABEL')
            print(current_label)
            print(classes[cur],end='-')
            cur=detections['detection_scores'][i]

            print(cur,end=' ')
            print(detections['detection_boxes'][i], end=' ')
            x0=(detections['detection_boxes'][i][0]) #xmin
            y0=(detections['detection_boxes'][i][1]) #ymin
            x1=(detections['detection_boxes'][i][2]) #xmax
            y1=(detections['detection_boxes'][i][3]) #ymax
            print('x0,y0,x1,y1')
            print(x0,y0,x1,y1)
            curarea=(x1-x0)*(y1-y0)
            print('curarea')
            print(curarea)
            ok=1
            for j in range(0,i):
                if mark[j]==0:
                    continue
                x2=(detections['detection_boxes'][j][0]) #xmin
                y2=(detections['detection_boxes'][j][1]) #ymin
                x3=(detections['detection_boxes'][j][2]) #xmax
                y3=(detections['detection_boxes'][j][3]) #ymax
                x4=max(x0,x2) #larger value from two xmin
                y4=max(y0,y2) #larger value from two ymin
                x5=min(x1,x3) #smaller value from two xmax
                y5=min(y1,y3) #smaller value from two ymax
                print('x2,y2,x3,y3,x4,y4,x5,y5')
                print(x2,y2,x3,y3,x4,y4,x5,y5)
                previous_label = labels[detections['detection_classes'][j]]
                if(previous_label == 86):
                  previous_label = 64
                print('PREVIOUS_LABEL')
                print(previous_label)

                if x4>=x5 or y4>=y5:  #both xmin must be less than both xmax and ymin <= ymax , otherwise no overlap
                    continue
                print("EKHANE ASHCHI") # we are here means overlapping case
                prevarea=(x3-x2)*(y3-y2)
                print('prevarea')
                print(prevarea)
                commonarea=(x5-x4)*(y5-y4)
                print('commonarea')
                print(commonarea)
                ins1=curarea/commonarea # large common area means smaller ins
                ins2=prevarea/commonarea
                print('ins1,ins2')
                print(ins1,ins2)

                characters_Eligible_For_Vowels =[11,12,13,14,16,17,18,19,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,49,68,69,70,71,72,73,74,75,76,77,78,79,81,82,83,84,85]
                vowels = [61,62,91]
                probable_clash_with_chondrobindu = [21,22,82]
                if (((current_label in characters_Eligible_For_Vowels and previous_label in vowels) or (current_label in vowels and previous_label in characters_Eligible_For_Vowels) or
                 (current_label in probable_clash_with_chondrobindu and previous_label == 49) or (current_label == 49 and previous_label in probable_clash_with_chondrobindu)) and
                 (current_label != previous_label)):
                    if(ins1<0.5 or ins2<0.5):
                        ok=0
                        cur=detections['detection_classes'][j]
                        print('classes[cur]')
                        print(classes[cur])
                        break
                else:
                  print("TUKI3")
                  if(ins1<2 or ins2<2): #
                        ok=0
                        cur=detections['detection_classes'][j]
                        print('classes[cur]')
                        print(classes[cur])
                        break
            if ok==1:
                mark[i]=1
                cur=detections['detection_classes'][i]
                print("detections['detection_classes'][i]")
                print(cur)
            print(ok)
        for i in range(0,num_detections):
            if mark[i]==0:
                continue
            cur=detections['detection_classes'][i]
            cur=classes[cur]
            y0=(detections['detection_boxes'][i][1])
            pair = (y0,cur)
            myletters.append(pair)
        myletters.sort(key = lambda x: x[0])
        res_list = [x[1] for x in myletters]
        print(res_list)

        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='ো':
              res_list.pop(i)
            if x=='ে' or x=='ি' or x=='ৈ' or x=='ৌ'or x=='্র' :
                res_list[i],res_list[i+1]=res_list[i+1],res_list[i]
        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='অ' and y=='া':
                print('yo')
                res_list[i]='আ'
                res_list.pop(i+1)
        print(res_list)
        for i in res_list:
            print(i,end='')
            final_Lekha.append(i)
        final_Lekha.append(" ")

        label_id_offset = 1
        image_np_with_detections = image_np.copy()
        row,col,dummy=image_np.shape
        list_with_all_boxes = []
        if(isRealTime):
            cv2.imshow('object detection',  cv2.resize(image_np_with_detections, (800, 600)))
        else:
            plt.figure(num=None, figsize=(20,20), dpi=40, facecolor='w', edgecolor='k')
            plt.imshow(image_np_with_detections, interpolation='nearest')
            plt.gca().add_patch(Rectangle((10,60),10,10,edgecolor='r',facecolor='None'))
            cntr=0
            for i in range(0,num_detections):
                if mark[i]==0:
                    continue
                classname=detections['detection_classes'][i]
                classname= labels[classname]
                x0=(detections['detection_boxes'][i][0])*row
                y0=(detections['detection_boxes'][i][1])*col
                x1=(detections['detection_boxes'][i][2])*row
                y1=(detections['detection_boxes'][i][3])*col
                if cntr==0:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='r',facecolor='None'))
                    cntr=1
                elif cntr==1:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='b',facecolor='None'))
                    cntr=2
                elif cntr==2:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='k',facecolor='None'))
                    cntr=3
                elif cntr==3:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='g',facecolor='None'))
                    cntr=0
                list_with_single_boxes = [x0,y0,x1,y1,classname]
                list_with_all_boxes.append(list_with_single_boxes)

        return list_with_all_boxes

Ablation: `no_pno`. This cell activates only when EXPERIMENT selects it.

In [ ]:
if EXPERIMENT == 'no_pno':
    """
    Object Detection (On Image) From TF2 Saved Model
    =====================================
    """
    final_Lekha = []
    def detect_frame(frame,IMAGE_PATHS,isRealTime = False):
        import os
        os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'    # Suppress TensorFlow logging (1)
        import pathlib
        import tensorflow as tf
        import cv2
        import argparse
        import numpy as np


        # PROVIDE PATH TO IMAGE DIRECTORY
        print(IMAGE_PATHS)
        IMAGE_PATHS = f'{IMAGE_PATHS}'

        if frame is None:
            raise ValueError(f"Cannot read image: {IMAGE_PATHS}")
        image_np = np.array(frame)
        # PROVIDE PATH TO MODEL DIRECTORY
        PATH_TO_MODEL_DIR = str(SSD_MODEL_DIR)

        # PROVIDE PATH TO LABEL MAP

        # PROVIDE THE MINIMUM CONFIDENCE THRESHOLD
        MIN_CONF_THRESH = float(0.20)

        # LOAD THE MODEL

        import time
        from object_detection.utils import label_map_util
        from object_detection.utils import visualization_utils as viz_utils

        PATH_TO_SAVED_MODEL = str(Path(PATH_TO_MODEL_DIR) / 'saved_model')

        print('Loading model...', end='')
        start_time = time.time()

        # LOAD SAVED MODEL AND BUILD DETECTION FUNCTION
        if PATH_TO_SAVED_MODEL not in _saved_models:
            if not (Path(PATH_TO_SAVED_MODEL) / 'saved_model.pb').is_file():
                raise FileNotFoundError(f'Provide the trained SavedModel: {PATH_TO_SAVED_MODEL}')
            _saved_models[PATH_TO_SAVED_MODEL] = tf.saved_model.load(PATH_TO_SAVED_MODEL)
        detect_fn = _saved_models[PATH_TO_SAVED_MODEL]

        end_time = time.time()
        elapsed_time = end_time - start_time
        print('Done! Took {} seconds'.format(elapsed_time))

        # LOAD LABEL MAP DATA FOR PLOTTING

        category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS,
                                                                            use_display_name=True)

        import numpy as np
        from PIL import Image
        import matplotlib.pyplot as plt
        import warnings


        def load_image_into_numpy_array(path):
            """Load an image from file into a numpy array.
            Puts image into numpy array to feed into tensorflow graph.
            Note that by convention we put it into a numpy array with shape
            (height, width, channels), where channels=3 for RGB.
            Args:
              path: the file path to the image
            Returns:
              uint8 numpy array with shape (img_height, img_width, 3)
            """
            return np.array(Image.open(path))

        print('Running inference for {}... '.format(IMAGE_PATHS), end='')

        image = cv2.imread(IMAGE_PATHS)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {IMAGE_PATHS}")
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_expanded = np.expand_dims(image_rgb, axis=0)

        # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
        # Preserve original BGR input. Binary word images have identical channels.
        input_tensor = tf.convert_to_tensor(image)
        # The model expects a batch of images, so add an axis with `tf.newaxis`.
        input_tensor = input_tensor[tf.newaxis, ...]

        detections = detect_fn(input_tensor)
        validate_detections(detections)
        temp__num_detection = 0
        for i in range(min(100, int(detections['num_detections'][0]))):
          if detections['detection_scores'][0][i]>0.2 :
            temp__num_detection+=1
        print(temp__num_detection)

        detections['num_detections'] = temp__num_detection

        # All outputs are batches tensors.
        # Convert to numpy arrays, and take index [0] to remove the batch dimension.
        # We're only interested in the first num_detections.
        num_detections = int(detections.pop('num_detections'))

        detections = {key: value[0, :num_detections].numpy()
                      for key, value in detections.items()}
        detections['num_detections'] = num_detections
        print('hoye ja')
        print(detections['detection_classes'])

        # detection_classes should be ints.
        detections['detection_classes'] = detections['detection_classes'].astype(np.int64)

        image_with_detections = image.copy()

        # SET MIN_SCORE_THRESH BASED ON YOU MINIMUM THRESHOLD FOR DETECTIONS
        viz_utils.visualize_boxes_and_labels_on_image_array(
              image_with_detections,
              detections['detection_boxes'],
              detections['detection_classes'],
              detections['detection_scores'],
              category_index,
              use_normalized_coordinates=True,
              max_boxes_to_draw=100,
              min_score_thresh=0.2,
              agnostic_mode=False)

        print('Done')
        print(num_detections)

        print(detections['detection_classes'])
        print(detections['detection_scores'])
        cv2_imshow(image_with_detections)

        mark = [0]*num_detections
        myletters = []
        for i in range(0,num_detections):
            cur=detections['detection_classes'][i]
            current_label = labels[cur]
            if(current_label == 86):
              current_label = 64
            print('CURRENT_LABEL')
            print(current_label)
            print(classes[cur],end='-')
            cur=detections['detection_scores'][i]

            print(cur,end=' ')
            print(detections['detection_boxes'][i], end=' ')
            x0=(detections['detection_boxes'][i][0]) #xmin
            y0=(detections['detection_boxes'][i][1]) #ymin
            x1=(detections['detection_boxes'][i][2]) #xmax
            y1=(detections['detection_boxes'][i][3]) #ymax
            print('x0,y0,x1,y1')
            print(x0,y0,x1,y1)
            curarea=(x1-x0)*(y1-y0)
            print('curarea')
            print(curarea)
            ok=1
            for j in range(0,i):
                if mark[j]==0:
                    continue
                x2=(detections['detection_boxes'][j][0]) #xmin
                y2=(detections['detection_boxes'][j][1]) #ymin
                x3=(detections['detection_boxes'][j][2]) #xmax
                y3=(detections['detection_boxes'][j][3]) #ymax
                x4=max(x0,x2) #larger value from two xmin
                y4=max(y0,y2) #larger value from two ymin
                x5=min(x1,x3) #smaller value from two xmax
                y5=min(y1,y3) #smaller value from two ymax
                print('x2,y2,x3,y3,x4,y4,x5,y5')
                print(x2,y2,x3,y3,x4,y4,x5,y5)
                previous_label = labels[detections['detection_classes'][j]]
                if(previous_label == 86):
                  previous_label = 64
                print('PREVIOUS_LABEL')
                print(previous_label)

                if x4>=x5 or y4>=y5:  #both xmin must be less than both xmax and ymin <= ymax , otherwise no overlap
                    continue
                print("EKHANE ASHCHI") # we are here means overlapping case
                prevarea=(x3-x2)*(y3-y2)
                print('prevarea')
                print(prevarea)
                commonarea=(x5-x4)*(y5-y4)
                print('commonarea')
                print(commonarea)
                ins1=curarea/commonarea # large common area means smaller ins
                ins2=prevarea/commonarea
                print('ins1,ins2')
                print(ins1,ins2)

                print("TUKI3")
                if(ins1<2 or ins2<2): #
                      ok=0
                      cur=detections['detection_classes'][j]
                      print('classes[cur]')
                      print(classes[cur])
                      break
            if ok==1:
                mark[i]=1
                cur=detections['detection_classes'][i]
                print("detections['detection_classes'][i]")
                print(cur)
            print(ok)
        for i in range(0,num_detections):
            if mark[i]==0:
                continue
            cur=detections['detection_classes'][i]
            cur=classes[cur]
            y0=(detections['detection_boxes'][i][1])
            pair = (y0,cur)
            myletters.append(pair)
        myletters.sort(key = lambda x: x[0])
        res_list = [x[1] for x in myletters]
        print(res_list)

        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='ো':
              res_list.pop(i)
            if x=='ে' or x=='ি' or x=='ৈ' or x=='ৌ':
                res_list[i],res_list[i+1]=res_list[i+1],res_list[i]
        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='অ' and y=='া':
                print('yo')
                res_list[i]='আ'
                res_list.pop(i+1)
        print(res_list)
        for i in res_list:
            print(i,end='')
            final_Lekha.append(i)
        final_Lekha.append(" ")

        label_id_offset = 1
        image_np_with_detections = image_np.copy()
        row,col,dummy=image_np.shape
        list_with_all_boxes = []
        if(isRealTime):
            cv2.imshow('object detection',  cv2.resize(image_np_with_detections, (800, 600)))
        else:
            plt.figure(num=None, figsize=(20,20), dpi=40, facecolor='w', edgecolor='k')
            plt.imshow(image_np_with_detections, interpolation='nearest')
            plt.gca().add_patch(Rectangle((10,60),10,10,edgecolor='r',facecolor='None'))
            cntr=0
            for i in range(0,num_detections):
                if mark[i]==0:
                    continue
                classname=detections['detection_classes'][i]
                classname= labels[classname]
                x0=(detections['detection_boxes'][i][0])*row
                y0=(detections['detection_boxes'][i][1])*col
                x1=(detections['detection_boxes'][i][2])*row
                y1=(detections['detection_boxes'][i][3])*col
                if cntr==0:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='r',facecolor='None'))
                    cntr=1
                elif cntr==1:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='b',facecolor='None'))
                    cntr=2
                elif cntr==2:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='k',facecolor='None'))
                    cntr=3
                elif cntr==3:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='g',facecolor='None'))
                    cntr=0
                list_with_single_boxes = [x0,y0,x1,y1,classname]
                list_with_all_boxes.append(list_with_single_boxes)

        return list_with_all_boxes

Ablation: `no_nms_pno`. This cell activates only when EXPERIMENT selects it.

In [ ]:
if EXPERIMENT == 'no_nms_pno':
    """
    Object Detection (On Image) From TF2 Saved Model
    =====================================
    """
    final_Lekha = []
    def detect_frame(frame,IMAGE_PATHS,isRealTime = False):
        import os
        os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'    # Suppress TensorFlow logging (1)
        import pathlib
        import tensorflow as tf
        import cv2
        import argparse
        import numpy as np


        # PROVIDE PATH TO IMAGE DIRECTORY
        print(IMAGE_PATHS)
        IMAGE_PATHS = f'{IMAGE_PATHS}'

        if frame is None:
            raise ValueError(f"Cannot read image: {IMAGE_PATHS}")
        image_np = np.array(frame)
        # PROVIDE PATH TO MODEL DIRECTORY
        PATH_TO_MODEL_DIR = str(SSD_MODEL_DIR)

        # PROVIDE PATH TO LABEL MAP

        # PROVIDE THE MINIMUM CONFIDENCE THRESHOLD
        MIN_CONF_THRESH = float(0.20)

        # LOAD THE MODEL

        import time
        from object_detection.utils import label_map_util
        from object_detection.utils import visualization_utils as viz_utils

        PATH_TO_SAVED_MODEL = str(Path(PATH_TO_MODEL_DIR) / 'saved_model')

        print('Loading model...', end='')
        start_time = time.time()

        # LOAD SAVED MODEL AND BUILD DETECTION FUNCTION
        if PATH_TO_SAVED_MODEL not in _saved_models:
            if not (Path(PATH_TO_SAVED_MODEL) / 'saved_model.pb').is_file():
                raise FileNotFoundError(f'Provide the trained SavedModel: {PATH_TO_SAVED_MODEL}')
            _saved_models[PATH_TO_SAVED_MODEL] = tf.saved_model.load(PATH_TO_SAVED_MODEL)
        detect_fn = _saved_models[PATH_TO_SAVED_MODEL]

        end_time = time.time()
        elapsed_time = end_time - start_time
        print('Done! Took {} seconds'.format(elapsed_time))

        # LOAD LABEL MAP DATA FOR PLOTTING

        category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS,
                                                                            use_display_name=True)

        import numpy as np
        from PIL import Image
        import matplotlib.pyplot as plt
        import warnings


        def load_image_into_numpy_array(path):
            """Load an image from file into a numpy array.
            Puts image into numpy array to feed into tensorflow graph.
            Note that by convention we put it into a numpy array with shape
            (height, width, channels), where channels=3 for RGB.
            Args:
              path: the file path to the image
            Returns:
              uint8 numpy array with shape (img_height, img_width, 3)
            """
            return np.array(Image.open(path))

        print('Running inference for {}... '.format(IMAGE_PATHS), end='')

        image = cv2.imread(IMAGE_PATHS)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {IMAGE_PATHS}")
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_expanded = np.expand_dims(image_rgb, axis=0)

        # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
        # Preserve original BGR input. Binary word images have identical channels.
        input_tensor = tf.convert_to_tensor(image)
        # The model expects a batch of images, so add an axis with `tf.newaxis`.
        input_tensor = input_tensor[tf.newaxis, ...]

        detections = detect_fn(input_tensor)
        validate_detections(detections)
        temp__num_detection = 0
        for i in range(min(100, int(detections['num_detections'][0]))):
          if detections['detection_scores'][0][i]>0.2 :
            temp__num_detection+=1
        print(temp__num_detection)

        detections['num_detections'] = temp__num_detection

        # All outputs are batches tensors.
        # Convert to numpy arrays, and take index [0] to remove the batch dimension.
        # We're only interested in the first num_detections.
        num_detections = int(detections.pop('num_detections'))

        detections = {key: value[0, :num_detections].numpy()
                      for key, value in detections.items()}
        detections['num_detections'] = num_detections
        print('hoye ja')
        print(detections['detection_classes'])

        # detection_classes should be ints.
        detections['detection_classes'] = detections['detection_classes'].astype(np.int64)

        image_with_detections = image.copy()

        # SET MIN_SCORE_THRESH BASED ON YOU MINIMUM THRESHOLD FOR DETECTIONS
        viz_utils.visualize_boxes_and_labels_on_image_array(
              image_with_detections,
              detections['detection_boxes'],
              detections['detection_classes'],
              detections['detection_scores'],
              category_index,
              use_normalized_coordinates=True,
              max_boxes_to_draw=100,
              min_score_thresh=0.2,
              agnostic_mode=False)

        print('Done')
        print(num_detections)

        print(detections['detection_classes'])
        print(detections['detection_scores'])
        cv2_imshow(image_with_detections)

        mark = [0]*num_detections
        myletters = []
        for i in range(0,num_detections):
            cur=detections['detection_classes'][i]
            current_label = labels[cur]
            if(current_label == 86):
              current_label = 64
            print('CURRENT_LABEL')
            print(current_label)
            print(classes[cur],end='-')
            cur=detections['detection_scores'][i]

            print(cur,end=' ')
            print(detections['detection_boxes'][i], end=' ')

            mark[i]=1
            cur=detections['detection_classes'][i]
            print("detections['detection_classes'][i]")
            print(cur)

        for i in range(0,num_detections):
            if mark[i]==0:
                continue
            cur=detections['detection_classes'][i]
            cur=classes[cur]
            y0=(detections['detection_boxes'][i][1])
            pair = (y0,cur)
            myletters.append(pair)
        myletters.sort(key = lambda x: x[0])
        res_list = [x[1] for x in myletters]
        print(res_list)

        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='ো':
              res_list.pop(i)
            if x=='ে' or x=='ি' or x=='ৈ' or x=='ৌ':
                res_list[i],res_list[i+1]=res_list[i+1],res_list[i]
        for i in range(len(res_list)-2, -1, -1):
            x=res_list[i]
            y=res_list[i+1]
            if x=='অ' and y=='া':
                print('yo')
                res_list[i]='আ'
                res_list.pop(i+1)
        print(res_list)
        for i in res_list:
            print(i,end='')
            final_Lekha.append(i)
        final_Lekha.append(" ")

        label_id_offset = 1
        image_np_with_detections = image_np.copy()
        row,col,dummy=image_np.shape
        list_with_all_boxes = []
        if(isRealTime):
            cv2.imshow('object detection',  cv2.resize(image_np_with_detections, (800, 600)))
        else:
            plt.figure(num=None, figsize=(20,20), dpi=40, facecolor='w', edgecolor='k')
            plt.imshow(image_np_with_detections, interpolation='nearest')
            plt.gca().add_patch(Rectangle((10,60),10,10,edgecolor='r',facecolor='None'))
            cntr=0
            for i in range(0,num_detections):
                if mark[i]==0:
                    continue
                classname=detections['detection_classes'][i]
                classname= labels[classname]
                x0=(detections['detection_boxes'][i][0])*row
                y0=(detections['detection_boxes'][i][1])*col
                x1=(detections['detection_boxes'][i][2])*row
                y1=(detections['detection_boxes'][i][3])*col
                if cntr==0:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='r',facecolor='None'))
                    cntr=1
                elif cntr==1:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='b',facecolor='None'))
                    cntr=2
                elif cntr==2:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='k',facecolor='None'))
                    cntr=3
                elif cntr==3:
                    plt.gca().add_patch(Rectangle((y0,x0),y1-y0,x1-x0,linewidth=3,edgecolor='g',facecolor='None'))
                    cntr=0
                list_with_single_boxes = [x0,y0,x1,y1,classname]
                list_with_all_boxes.append(list_with_single_boxes)

        return list_with_all_boxes

In [ ]:
import xml.etree.ElementTree as ET

def read_content(xml_file: str):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    list_with_all_boxes = []

    for boxes in root.iter('object'):

        filename = root.find('filename').text
        classname = None
        ymin, xmin, ymax, xmax = None, None, None, None
        classname = int(boxes.find("name").text)
        ymin = int(boxes.find("bndbox/ymin").text)
        xmin = int(boxes.find("bndbox/xmin").text)
        ymax = int(boxes.find("bndbox/ymax").text)
        xmax = int(boxes.find("bndbox/xmax").text)

        if not 1 <= classname <= 92:
            raise ValueError(f'Invalid class ID {classname} in {xml_file}')
        if xmin < 0 or ymin < 0 or xmax <= xmin or ymax <= ymin:
            raise ValueError(f'Invalid bounding box in {xml_file}')
        list_with_single_boxes = [xmin, ymin, xmax, ymax,classname]
        list_with_all_boxes.append(list_with_single_boxes)

    return list_with_all_boxes

In [ ]:
PATH_TO_TEST_IMAGES_DIR = Path(PATH_TO_TEST_IMAGES_DIR)
TEST_IMAGE_PATHS = sorted(PATH_TO_TEST_IMAGES_DIR.glob('*.bmp'))
if not TEST_IMAGE_PATHS:
    raise FileNotFoundError(f'No BMP test images in {PATH_TO_TEST_IMAGES_DIR}')
# Pair by filename, never by independent positions in two sorted lists.
TEST_XML_PATHS = [image_path.with_suffix('.xml') for image_path in TEST_IMAGE_PATHS]
for xml_path in TEST_XML_PATHS:
    if not xml_path.is_file():
        raise FileNotFoundError(f'Missing annotation: {xml_path}')
if {p.stem for p in PATH_TO_TEST_IMAGES_DIR.glob('*.xml')} != {p.stem for p in TEST_IMAGE_PATHS}:
    raise ValueError('Test folder contains XML files without matching BMP images.')
if not Path(PATH_TO_LABELS).is_file():
    raise FileNotFoundError(f'Missing original label map: {PATH_TO_LABELS}; see training label-map cell.')
from object_detection.utils import label_map_util
if label_map_util.get_label_map_dict(PATH_TO_LABELS) != {str(i): i for i in range(1, 93)}:
    raise ValueError('Label map must preserve the original numeric names and IDs 1..92.')
print(len(TEST_IMAGE_PATHS))

In [ ]:
TP = [0]*93
FP = [0]*93
FN = [0]*93
print(FP)

TP_OLD = [0]*93
FP_OLD = [0]*93
FN_OLD = [0]*93

In [ ]:
# A rerun starts a fresh evaluation instead of doubling the previous counts.
TP = [0]*93
FP = [0]*93
FN = [0]*93
TP_OLD = [0]*93
FP_OLD = [0]*93
FN_OLD = [0]*93
final_Lekha = []

confusion_matrix = [[0 for i in range(93)] for j in range(93)]
confusion_matrix_OLD = [[0 for i in range(93)] for j in range(93)]


for k in range(0,len(TEST_IMAGE_PATHS)):
    print("K NUMBER = ",k)
    image_path = TEST_IMAGE_PATHS[k]
    print(image_path)
    xml_path = TEST_XML_PATHS[k]
    print(xml_path)
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    cv2_imshow(frame)
    A = detect_frame(frame,image_path)
    plt.close("all")
    B = read_content(xml_path)
    print("A-----------------------------------------------------------")

    print(A)
    print(B)

    biou=0.5
    markA = [0]*len(A)
    mark=[0]*len(B)
    maxiou=0.4
    for j in range(0,len(B)):

        x2=B[j][0]
        y2=B[j][1]
        x3=B[j][2]
        y3=B[j][3]
        classB=B[j][4]
        if(classB == 86):
          classB = 64
        print("actual class",classB)
        OriginalArea=(x3-x2)*(y3-y2)
        got = 0
        for i in range(0,len(A)):
            if markA[i]==1:
              print("EKHANE")
              continue
            y0=A[i][0]
            x0=A[i][1]
            y1=A[i][2]
            x1=A[i][3]
            classA=A[i][4]
            if(classA == 86):
              classA = 64
            x4=max(x0,x2)
            y4=max(y0,y2)
            x5=min(x1,x3)
            y5=min(y1,y3)
            ComputedArea=(x1-x0)*(y1-y0)
            if x4>=x5 or y4>=y5:
                continue
            commonarea=(x5-x4)*(y5-y4)
            iou=commonarea/(ComputedArea+OriginalArea-commonarea)
            if x4>x5 or y4>y5:
                continue
            if x4>x5 or y4>y5:
                continue
            if iou>maxiou:
              got = 1
              mxx=i
              markA[mxx]=1
              if (classA == classB):
                TP[classA]=TP[classA]+1
                confusion_matrix[classB][classA] = confusion_matrix[classB][classA] + 1
                break
              else:
                FP[classA]=FP[classA]+1
                confusion_matrix[classB][classA] = confusion_matrix[classB][classA] + 1
                break

        if got==0:
          FN[classB]=FN[classB]+1
          confusion_matrix[classB][0] = confusion_matrix[classB][0] + 1
    for i in range(0,len(A)):
      if(markA[i]==0):
        classA=A[i][4]
        if(classA == 86):
          classA = 64
        FP[classA]=FP[classA]+1
        confusion_matrix[0][classA] = confusion_matrix[0][classA]+ 1

    for i in range (len(TP)):
      if(TP[i]!=TP_OLD[i]):
        print("TP CHANGE index = ",i," TP DIFFERENCE = ",[TP_OLD[i]-TP[i]])
      if(FP[i]!=FP_OLD[i]):
        print("FP CHANGE index = ",i," FP DIFFERENCE = ",[FP_OLD[i]-FP[i]])
      if(FN[i]!=FN_OLD[i]):
        print("FN CHANGE index = ",i," FN DIFFERENCE = ",[FN_OLD[i]-FN[i]])

    for m in range(0,93):
      for n in range(0,93):
        if (confusion_matrix [m][n] != confusion_matrix_OLD[m][n]):
          print ("confusion_matrix_row = ",classes[m]," COLUMN = ",classes[n]," DIFFERENCE = ",confusion_matrix [m][n] - confusion_matrix_OLD[m][n])
          confusion_matrix_OLD [m][n] = confusion_matrix[m][n]
    TP_OLD = TP.copy()
    FP_OLD = FP.copy()
    FN_OLD = FN.copy()

    print("TP = ",TP)
    print("FP = ",FP)
    print("FN = ",FN)
    print("TP_OLD = ",TP)
    print("FP_OLD = ",FP)
    print("FN_OLD = ",FN)
    print (" ")

Optional Levenshtein evaluation. Enable RUN_WORD_EVALUATION in the configuration.

In [ ]:
if RUN_WORD_EVALUATION:
    final_Lekha = []
    from functools import cmp_to_key
    correct_word_percentage = [0, 0, 0, 0, 0]

    def sortbyCond(a, b):
        print ("a = ", a)
        print ("b = ", b)

        if (a[0] != b[0]):
            print ( "a[0] - b[0]",a[0] - b[0])
            return (a[0] - b[0])
        else:
            print ("b[1] - a[1]",b[1] - a[1])
            return (b[1] - a[1])

    def levenshtein(a, b):
        """Calculates the Levenshtein distance between a and b."""
        n, m = len(a), len(b)
        if n > m:
            a, b = b, a
            n, m = m, n
        current = range(n + 1)
        for i in range(1, m + 1):
            previous, current = current, [i] + [0] * n
            for j in range(1, n + 1):
                add, delete = previous[j] + 1, current[(j - 1)] + 1
                change = previous[(j - 1)]
                if a[(j - 1)] != b[(i - 1)]:
                    change = change + 1
                current[j] = min(add, delete, change)

        return current[n]
    for k in range(0,len(TEST_IMAGE_PATHS)):
        print("K NUMBER = ",k)
        image_path = TEST_IMAGE_PATHS[k]
        print(image_path)
        xml_path = TEST_XML_PATHS[k]
        print(xml_path)
        frame = cv2.imread(str(image_path))
        if frame is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        cv2_imshow(frame)
        A = detect_frame(frame,image_path)
        plt.close("all")
        B = read_content(xml_path)
        arr1 = A
        arr2 = B
        sorted_a1 = []
        sorted_a2 = []
        for i in range (0,len(arr1)):
          sorted_a1.append([arr1[i][1], arr1[i][2]])
        for i in range (0,len(arr2)):
          sorted_a2.append([arr2[i][0], arr2[i][3]])
        sorted_a1.sort(key = cmp_to_key(sortbyCond))
        sorted_a2.sort(key = cmp_to_key(sortbyCond))

        C =[]
        D=[]

        for i in range(0,len(sorted_a1)):
          x = sorted_a1[i][0]
          y = sorted_a1[i][1]
          for j in range(0,len(arr1)):
            if x==arr1[j][1] and y==arr1[j][2]:
              C.append(arr1[j][4])
              break

        for i in range(0,len(sorted_a2)):
          x = sorted_a2[i][0]
          y = sorted_a2[i][1]
          for j in range(0,len(arr2)):
            if x==arr2[j][0] and y==arr2[j][3]:
              D.append(arr2[j][4])
              break
        print("EKHASNE SORTED THAKBE")
        print(C)
        print(D)
        val = levenshtein(C,D)
        print(val)

        per = (len(D)-val)/len(D) if D else (1.0 if not C else 0.0)
        per = per*100
        print(per)

        if (per==100.0):
          correct_word_percentage[0]+=1
        elif (per>=75.0):
          correct_word_percentage[1]+=1
        elif (per>=50.0):
          correct_word_percentage[2]+=1
        elif (per>=25.0):
          correct_word_percentage[3]+=1
        else:
          correct_word_percentage[4]+=1
        print(correct_word_percentage)

Special Thank you to :
M. S. Hasan and A. Pal, “Optical character recognition for handwritten bangla documents,” 2021, unpublished undergraduate thesis.

In case of need, many code snippets were collected from Hasan et al's work and then upgraded. Authors will be forever grateful to them.

Save results. Choose a new output folder for each experiment.

In [ ]:
import json
import hashlib
import importlib.metadata
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f'Choose an empty OUTPUT_DIR: {OUTPUT_DIR}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'recognized.txt').write_text(''.join(final_Lekha), encoding='utf-8')
(OUTPUT_DIR / 'results.json').write_text(json.dumps({
    'experiment': EXPERIMENT,
    'test_image_count': len(TEST_IMAGE_PATHS),
    'test_files': [{'image': str(image_path), 'xml': str(xml_path),
                    'image_sha256': hashlib.sha256(image_path.read_bytes()).hexdigest(),
                    'xml_sha256': hashlib.sha256(xml_path.read_bytes()).hexdigest()}
                   for image_path, xml_path in zip(TEST_IMAGE_PATHS, TEST_XML_PATHS)],
    'TP': TP, 'FP': FP, 'FN': FN, 'confusion_matrix': confusion_matrix,
    'word_accuracy_bins': correct_word_percentage if RUN_WORD_EVALUATION else None,
    'word_accuracy_bins_order': ['100', '75_to_less_than_100', '50_to_less_than_75', '25_to_less_than_50', 'less_than_25'],
    'metric_semantics': 'Original first IoU > 0.4 match; wrong-class match adds FP but not FN.',
    'tensorflow': tf.__version__, 'opencv': cv2.__version__,
    'numpy': np.__version__, 'python': sys.version,
    'saved_model_paths': sorted(_saved_models),
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Results saved in {OUTPUT_DIR}')